# Phase B — Cache frozen MERLIN features to GCS

Runs in Colab, writes features straight to a Google Cloud Storage bucket.
**Stream-and-delete:** download one CT volume → preprocess → encode with frozen MERLIN →
save ~1 MB of features to GCS → **delete the 420 MB scan**. Peak disk stays a few GB, and
your permanent GCP footprint is well under 1 GB instead of 250 GB.

Two things get cached per volume:

| | What | Used for |
|---|---|---|
| `pooled` | MERLIN's 512-d contrastive embedding | baselines (`v_cur`, `v_cur − v_pri`) |
| `grid` | `layer4` pre-pool feature map (~2048×5×7×7) | the difference transformer |

MERLIN's `AdaptiveAvgPool3d((1,1,1))` collapses all spatial detail into 512-d, so the
**grid is essential** — interval change lives in the spatial layout, not the pooled vector.

**Run this in a SEPARATE Colab session** from the labeling notebook (MedGemma-27B is using
~54 GB of GPU there). A T4 or L4 is plenty here — this is inference only.

**Order:** run cells 1–7 (verification) first. They prove CT-RATE access works, MERLIN
loads, and tell us the TRUE feature shape. Only then run cell 8 (production loop).

In [ ]:
# 1. Install deps (merlin-vlm pulls torch + monai + transformers)
!pip -q install merlin-vlm huggingface_hub google-cloud-storage nibabel
import torch, monai
print('torch', torch.__version__, '| monai', monai.__version__)
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 2. Auth: Google Cloud + Hugging Face
PROJECT_ID = 'neal-ct-temporal'            # dedicated project for this work
BUCKET     = 'neal-ct-temporal-features'   # already created in us-central1
REGION     = 'us-central1'

from google.colab import auth
auth.authenticate_user()                # browser popup; no service-account key needed
!gcloud config set project {PROJECT_ID} -q

# create the bucket if it doesn't exist (ignore error if it already does)
!gsutil ls -b gs://{BUCKET} 2>/dev/null || gsutil mb -l {REGION} gs://{BUCKET}
!gsutil ls -b gs://{BUCKET}

from huggingface_hub import notebook_login
notebook_login()                        # CT-RATE is gated: accept its license on HF first
print('\nNOTE: CT-RATE has its OWN license, separate from MedGemma.')
print('Accept at https://huggingface.co/datasets/ibrahimhamamci/CT-RATE')

In [ ]:
# 3. CT-RATE volume -> HF path, and a selective downloader (self-contained)
import os, time, gc, json, csv
csv.field_size_limit(10**9)
REPO_ID = 'ibrahimhamamci/CT-RATE'
VOL_DIR = '/content/volumes'
os.makedirs(VOL_DIR, exist_ok=True)

def remote_candidates(vol):
    """'train_3_b_1.nii.gz' -> candidate HF paths. CT-RATE nests split/patient/scan/file,
    and ships corrected '<split>_fixed' folders which we try first."""
    base = vol.replace('.nii.gz','').replace('.nii','')
    p = base.split('_')
    split, pid, scan = p[0], p[1], p[2]
    patient, scan_folder = f'{split}_{pid}', f'{split}_{pid}_{scan}'
    folders = [f'{split}_fixed', split] if split in ('train','valid') else [split]
    return [f'dataset/{sf}/{patient}/{scan_folder}/{vol}' for sf in folders]

def download_volume(vol):
    """Fetch ONE volume. Returns local path or None."""
    from huggingface_hub import hf_hub_download
    last = None
    for path in remote_candidates(vol):
        try:
            return hf_hub_download(REPO_ID, path, repo_type='dataset', local_dir=VOL_DIR)
        except Exception as e:
            last = e
    print(f'    !! could not fetch {vol}: {str(last)[:120]}')
    return None
print('downloader ready')

In [ ]:
# 4. VERIFY (a): can we actually download a CT-RATE volume?
t0 = time.time()
fp = download_volume('train_3_a_1.nii.gz')
if fp:
    mb = os.path.getsize(fp)/1e6
    print(f'OK  {fp}\n    {mb:.0f} MB in {time.time()-t0:.0f}s')
    print(f'    -> at this size, 600 volumes = ~{600*mb/1000:.0f} GB if we kept them')
else:
    print('FAILED. Most likely: CT-RATE license not accepted, or folder layout differs.')

In [ ]:
# 5. Preprocessing (MERLIN's recipe) — MONAI transforms
from monai.transforms import (Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
                              Spacingd, ScaleIntensityRanged, SpatialPadd,
                              CenterSpatialCropd, ToTensord)
DEPTH = 160     # MERLIN spec: 224 x 224 x 160

PREP = Compose([
    LoadImaged(keys=['image']),
    EnsureChannelFirstd(keys=['image']),
    Orientationd(keys=['image'], axcodes='RAS'),
    Spacingd(keys=['image'], pixdim=(1.5, 1.5, 3), mode='bilinear'),   # 1.5mm in-plane, 3mm out
    ScaleIntensityRanged(keys=['image'], a_min=-1000, a_max=1000,
                        b_min=0.0, b_max=1.0, clip=True),              # HU window -> [0,1]
    SpatialPadd(keys=['image'], spatial_size=[224, 224, DEPTH]),
    CenterSpatialCropd(keys=['image'], roi_size=[224, 224, DEPTH]),
    ToTensord(keys=['image']),
])

def preprocess(path):
    """nii.gz path -> (1, 1, 224, 224, DEPTH) tensor ready for encode_image."""
    d = PREP({'image': path})
    return d['image'].unsqueeze(0)   # add batch dim
print(f'preprocessing ready (224 x 224 x {DEPTH})')

In [ ]:
# 6. Load frozen MERLIN + hook layer4 (the pre-pool grid we need)
from merlin import Merlin
import torch.nn.functional as F

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Merlin()
model.eval().to(DEVICE)
for p in model.parameters():
    p.requires_grad = False        # FROZEN, always
arch = model.model                 # exposes encode_image / encode_text

def find_i3res(a):
    """Locate the i3D-ResNet-152 module (owns contrastive_head + layer4)."""
    for _n, m in a.named_modules():
        if hasattr(m, 'contrastive_head') and hasattr(m, 'layer4'):
            return m
    return None

enc = find_i3res(arch)
assert enc is not None, 'could not locate the i3res image encoder'

_grab = {}
def _hook(mod, inp, out):
    _grab['grid'] = out.detach()
enc.layer4.register_forward_hook(_hook)

@torch.no_grad()
def encode(x):
    """-> (pooled 512-d, grid CxDxHxW). Grid comes from the layer4 hook."""
    _grab.clear()
    pooled, _ehr = arch.encode_image(x.to(DEVICE))
    if pooled.dim() == 1:
        pooled = pooled.unsqueeze(0)
    return pooled[0].float().cpu(), _grab['grid'][0].float().cpu()

print('MERLIN loaded (frozen), layer4 hook attached')

In [ ]:
# 7. VERIFY (b): TRUE feature shapes + timing + GCS round-trip
import numpy as np
from google.cloud import storage
gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)

t0 = time.time(); x = preprocess(fp); t_prep = time.time()-t0
print(f'preprocessed input : {tuple(x.shape)}   ({t_prep:.1f}s)')

t0 = time.time(); pooled, grid = encode(x); t_enc = time.time()-t0
print(f'pooled embedding   : {tuple(pooled.shape)}')
print(f'layer4 GRID        : {tuple(grid.shape)}   <-- the number the model design depends on')
n_tok = int(np.prod(grid.shape[1:]))
print(f'   -> {grid.shape[0]} channels x {n_tok} spatial tokens')
print(f'   -> fp16 size: {grid.numel()*2/1e6:.2f} MB/volume  '
      f'(600 volumes = {600*grid.numel()*2/1e9:.2f} GB)')
print(f'encode time        : {t_enc:.1f}s')
print(f'ESTIMATE for 600 volumes: ~{600*(t_prep+t_enc)/3600:.1f} h (+ download time)')

# GCS round-trip test
import io
buf = io.BytesIO()
np.savez_compressed(buf, pooled=pooled.numpy().astype('float16'),
                    grid=grid.numpy().astype('float16'))
buf.seek(0)
bucket.blob('features/_roundtrip_test.npz').upload_from_file(buf)
back = np.load(io.BytesIO(bucket.blob('features/_roundtrip_test.npz')
                          .download_as_bytes()))
print(f"\nGCS round-trip OK: pooled{back['pooled'].shape} grid{back['grid'].shape}")

In [ ]:
# 8. PRODUCTION LOOP — upload a pair manifest, then stream-and-delete
#    Upload subset_pairs.csv (preferred) or ctrate_pairs_enriched_v2.csv
from google.colab import files
up = files.upload()
MAN = list(up.keys())[0]
with open(MAN, newline='', encoding='utf-8') as f:
    pairs = list(csv.DictReader(f))
LIMIT = 10          # <-- start with 10 for a smoke test; raise once happy
pairs = pairs if LIMIT == 0 else pairs[:LIMIT]

# unique volumes (prior + current), deduped
vols = []
for r in pairs:
    vols += [r['prior_volume'], r['curr_volume']]
vols = list(dict.fromkeys(vols))

# resume: skip anything already in the bucket
have = {b.name.split('/')[-1].replace('.npz','')
        for b in gcs.list_blobs(BUCKET, prefix='features/')}
todo = [v for v in vols if v.replace('.nii.gz','') not in have]
print(f'{len(pairs)} pairs -> {len(vols)} volumes; {len(have)} already cached; {len(todo)} to do\n')

ok = fail = 0
t_start = time.time()
for i, vol in enumerate(todo, 1):
    try:
        fp = download_volume(vol)
        if fp is None:
            fail += 1; continue
        x = preprocess(fp)
        pooled, grid = encode(x)
        buf = io.BytesIO()
        np.savez_compressed(buf, pooled=pooled.numpy().astype('float16'),
                            grid=grid.numpy().astype('float16'))
        buf.seek(0)
        bucket.blob(f"features/{vol.replace('.nii.gz','')}.npz").upload_from_file(buf)
        ok += 1
    except Exception as e:
        print(f'  !! {vol}: {str(e)[:140]}'); fail += 1
    finally:
        # ALWAYS delete the raw scan -> keeps disk flat
        try:
            if fp and os.path.exists(fp): os.remove(fp)
        except Exception: pass
        del x, pooled, grid
        gc.collect(); torch.cuda.empty_cache()
    if i % 5 == 0 or i == len(todo):
        el = time.time()-t_start
        print(f'  {i}/{len(todo)}  ok={ok} fail={fail}  '
              f'{el/i:.0f}s/vol  ETA {(len(todo)-i)*el/i/60:.0f} min', flush=True)

print(f'\nDONE. cached={ok} failed={fail}')
!du -sh /content/volumes 2>/dev/null || true
!gsutil du -sh gs://{BUCKET}/features

In [ ]:
# 8b. PARALLEL caching (use INSTEAD of cell 8) — threaded downloads overlap GPU encode.
#     ~2x faster: a thread pool prefetches the next volumes while MERLIN encodes the
#     current one. Same stream-and-delete + bucket-resume as cell 8.
import concurrent.futures as cf, threading, queue
from google.colab import files
up = files.upload()                 # subset_pairs.csv
MAN = list(up.keys())[0]
with open(MAN, newline='', encoding='utf-8') as f:
    pairs = list(csv.DictReader(f))
LIMIT = 10          # smoke test first; set 0 for the full subset
pairs = pairs if LIMIT == 0 else pairs[:LIMIT]

vols = []
for r in pairs:
    vols += [r['prior_volume'], r['curr_volume']]
vols = list(dict.fromkeys(vols))

have = {b.name.split('/')[-1].replace('.npz','')
        for b in gcs.list_blobs(BUCKET, prefix='features/')}
todo = [v for v in vols if v.replace('.nii.gz','') not in have]
print(f'{len(pairs)} pairs -> {len(vols)} volumes; {len(have)} cached; {len(todo)} to do\n')

NWORKERS = 6                                  # parallel HF downloads
ready = queue.Queue(maxsize=NWORKERS * 2)     # (vol, local_path | None)

def _producer(vlist):
    with cf.ThreadPoolExecutor(max_workers=NWORKERS) as ex:
        futs = {ex.submit(download_volume, v): v for v in vlist}
        for fut in cf.as_completed(futs):
            v = futs[fut]
            try:
                ready.put((v, fut.result()))
            except Exception as e:
                print(f'  !! download {v}: {str(e)[:100]}'); ready.put((v, None))
    ready.put((None, None))                   # sentinel

threading.Thread(target=_producer, args=(todo,), daemon=True).start()

ok = fail = done = 0
t_start = time.time()
while True:
    vol, fp = ready.get()
    if vol is None:
        break
    done += 1
    try:
        if fp is None:
            fail += 1; continue
        x = preprocess(fp)
        pooled, grid = encode(x)
        buf = io.BytesIO()
        np.savez_compressed(buf, pooled=pooled.numpy().astype('float16'),
                            grid=grid.numpy().astype('float16'))
        buf.seek(0)
        bucket.blob(f"features/{vol.replace('.nii.gz','')}.npz").upload_from_file(buf)
        ok += 1
    except Exception as e:
        print(f'  !! encode {vol}: {str(e)[:120]}'); fail += 1
    finally:
        try:
            if fp and os.path.exists(fp): os.remove(fp)
        except Exception: pass
        for _v in ('x', 'pooled', 'grid'):
            globals().pop(_v, None)
        gc.collect(); torch.cuda.empty_cache()
    if done % 5 == 0 or done == len(todo):
        el = time.time() - t_start
        print(f'  {done}/{len(todo)}  ok={ok} fail={fail}  '
              f'{el/done:.0f}s/vol  ETA {(len(todo)-done)*el/done/60:.0f} min', flush=True)

print(f'\nDONE. cached={ok} failed={fail}')
!du -sh /content/volumes 2>/dev/null || true
!gsutil du -sh gs://{BUCKET}/features

In [ ]:
# 9. Cache the TEXT side too (frozen MERLIN text encoder) — seconds, ~KB
#    The 3-class progression prompts your difference embedding is matched against.
CANON = ['Medical material','Arterial wall calcification','Cardiomegaly',
  'Pericardial effusion','Coronary artery wall calcification','Hiatal hernia',
  'Lymphadenopathy','Emphysema','Atelectasis','Lung nodule','Lung opacity',
  'Pulmonary fibrotic sequela','Pleural effusion','Mosaic attenuation pattern',
  'Peribronchial thickening','Consolidation','Bronchiectasis','Interlobular septal thickening']

# two template families + paraphrases (ensembled by averaging)
TEMPLATES = {
  'worsened': ['there is more {f} than on the prior study',
               '{f} has increased compared to the prior study',
               'the {f} is larger than before',
               'interval increase in {f}'],
  'stable':   ['{f} is unchanged from the prior study',
               '{f} is stable compared to the prior study',
               'no significant change in {f}'],
  'improved': ['there is less {f} than on the prior study',
               '{f} has decreased compared to the prior study',
               'the {f} is smaller than before',
               'interval regression of {f}'],
}

@torch.no_grad()
def embed_texts(texts):
    t = arch.encode_text(texts)
    if t.dim() == 1: t = t.unsqueeze(0)
    return F.normalize(t.float().cpu(), dim=-1)

bank, raw = {}, {}
for f in CANON:
    fl = f[0].lower() + f[1:]
    for cls, temps in TEMPLATES.items():
        strs = [t.format(f=fl) for t in temps]
        embs = embed_texts(strs)
        bank[f'{f}|{cls}'] = F.normalize(embs.mean(0), dim=-1).numpy().astype('float16')
        raw[f'{f}|{cls}'] = strs

buf = io.BytesIO(); np.savez_compressed(buf, **bank); buf.seek(0)
bucket.blob('text/prompt_bank_3.npz').upload_from_file(buf)
bucket.blob('text/prompt_bank_3_strings.json').upload_from_string(json.dumps(raw, indent=2))
print(f'cached {len(bank)} prompt embeddings (18 findings x 3 classes) -> gs://{BUCKET}/text/')

# ---- SEPARATION CHECK: can the text encoder tell the 3 classes apart? ----
print('\nPer-finding 3x3 cosine (worsened/stable/improved). Off-diagonals should be << 1.0')
import itertools
classes = ['worsened','stable','improved']
offdiags = []
for f in CANON[:4]:
    V = np.stack([bank[f'{f}|{c}'] for c in classes]).astype('float32')
    M = V @ V.T
    print(f'  {f}:  w-s={M[0,1]:.3f}  w-i={M[0,2]:.3f}  s-i={M[1,2]:.3f}')
for f in CANON:
    V = np.stack([bank[f'{f}|{c}'] for c in classes]).astype('float32')
    M = V @ V.T
    offdiags += [M[0,1], M[0,2], M[1,2]]
m = float(np.mean(offdiags))
print(f'\nmean off-diagonal similarity across all 18 findings: {m:.3f}')
print('  < 0.90  -> classes are separable, cosine matching will work')
print('  > 0.98  -> prompts nearly identical; we need a trainable text projection')

## How to run this

**Verification first (cells 1–7).** These answer the three unknowns that could force a
design change, and they cost ~5 minutes and one 420 MB download:
1. Does CT-RATE download work? (license + folder layout)
2. Does MERLIN load, and what is the **true** `layer4` grid shape?
3. How long does one volume take, and does the GCS round-trip work?

**Then cell 9** — the text side. It's seconds of compute, and the **separation check at the
bottom is the single most important number right now**: if the 3 class prompts per finding
are nearly identical vectors under MERLIN's text encoder, cosine matching cannot
distinguish them no matter how good the image module is, and we'd add a small trainable
projection on the text side.

**Then cell 8 (sequential) or cell 8b (parallel) with `LIMIT = 10`** as a smoke test, and
finally `LIMIT = 0` for the full subset. Both are resumable — they list the bucket and skip
volumes already cached, so a disconnect costs nothing. Cell 8b overlaps HF downloads with
GPU encoding (`NWORKERS=6`) and is ~2x faster (~2h vs ~4h for the full subset).

**Manifest = `data/ctrate/subset_pairs.csv`** — the GOLD, 3-class-balanced subset produced by
`scripts/20_select_gold_subset.py` from the v3 labels: 600 pairs → 1,147 unique volumes
(~482 GB streamed-and-deleted), direction-balanced to ~34/36/29 (worsened/stable/improved),
patient-level 70/15/15 split. v3 labels are final, so this is ready to run.